In [0]:
%run "../includes/librerias"

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
#DataFrames con la data con la cual se va a trabajar, incluye filtros y campos

#DataFrame movies filtrado por el año 200 en adelante
movies_df = spark.read.table("movie_silver.movies")
movies_df = movies_df.filter(
                            (col("year_release_date") >= 2000)
                           )\
                     .select(movies_df.movie_id, movies_df.title, movies_df.duration_time, movies_df.release_date, movies_df.vote_average, movies_df.year_release_date)

#DataFrame movie languaje, para hacer join con languaje
movie_languages_df = spark.read.table("movie_silver.movies_languages")
movie_languages_df =movie_languages_df.select(movie_languages_df.movie_id, movie_languages_df.language_id)

language_df = spark.read.table("movie_silver.languages")
language_df = language_df.select(language_df.language_id, language_df.language_name)

#DataFrame genree
movie_genre_df = spark.read.table("movie_silver.movies_genres")
movie_genre_df = movie_genre_df.select(movie_genre_df.movie_id, movie_genre_df.genre_id)

genre_df = spark.read.table("movie_silver.genres")
genre_df = genre_df.select(genre_df.genre_id, genre_df.genre_name)



In [0]:
#Validamos los dataframes
movies_df.show(4)
movie_languages_df.show(4)
language_df.show(4)
movie_genre_df.show(4)
genre_df.show(4)


In [0]:
#Join entre movie y language

movies_movie_languages_df = movies_df.join(movie_languages_df,
                                           movies_df.movie_id == movie_languages_df.movie_id
                                           , "inner")\
                                      .select(movies_df["*"], movie_languages_df.language_id)#\
                                      #.filter(movie_languages_df.movie_id.isNull())

#display(movies_movie_languages_df)

movies_languages_df = movies_movie_languages_df.join(language_df,
                                           movies_movie_languages_df.language_id == language_df.language_id
                                           , "inner")\
                                      .select(movies_movie_languages_df["*"], language_df.language_name)#\
                                      #.filter(language_df.language_id.isNull())

display(movies_languages_df)



In [0]:
#Join entre movie y language

movies_movie_genre_df = movies_languages_df.join(movie_genre_df,
                                                 movies_languages_df.movie_id == movie_genre_df.movie_id
                                                ,"inner")\
                                           .select(movies_languages_df["*"], movie_genre_df.genre_id)#\
                                           #.filter(movie_genre_df.movie_id.isNull())
#hay datos nulos en genre
#display(movies_movie_genre_df)

movies_languages_genre_df = movies_movie_genre_df.join(genre_df,
                                                       movies_movie_genre_df.genre_id == genre_df.genre_id
                                                       , "inner")\
                                                 .select(movies_movie_genre_df["*"], genre_df.genre_name)#\
                                                 #.filter(movie_languages_df.movie_id.isNull())

display(movies_languages_genre_df)


In [0]:
#Seleccionamos las columnas y ordenamos

results_movie_genre_language_df = movies_languages_genre_df.select( movies_languages_genre_df.title,
                                                                    movies_languages_genre_df.duration_time,
                                                                    movies_languages_genre_df.release_date,
                                                                    movies_languages_genre_df.vote_average,
                                                                    movies_languages_genre_df.language_name,
                                                                    movies_languages_genre_df.genre_name
                                                                )\
                                                            .orderBy(movies_languages_genre_df.release_date.desc())


results_movie_genre_language_df = add_ingestion_date(results_movie_genre_language_df)
results_movie_genre_language_df = add_env(results_movie_genre_language_df)

display(results_movie_genre_language_df)

In [0]:
#Guardamos en la capa gold 

results_movie_genre_language_df.write.mode("overwrite").format("delta").saveAsTable("movie_gold.results_movie_genre_language")

#df = spark.read.parquet(f"{gold_folder_path}/results_movie_genre_language")
#display(df)

In [0]:
%sql
select * from movie_gold.results_movie_genre_language